In [1]:
import pandas as pd
import pulp
import numpy as np

In [2]:
# 1. Load the Datasets
df_items = pd.read_csv("mto_item_master.csv")
df_machines = pd.read_csv("machine_capacities.csv")
df_routings = pd.read_csv("production_routings.csv")
df_quotes = pd.read_csv("external_supplier_quotes.csv")
df_orders = pd.read_csv("active_indents.csv")

In [3]:
# 2. Pre-Processing & Merging Data
# Merge orders with item master and supplier quotes
df = df_orders.merge(df_items, on="item_id").merge(df_quotes, on="item_id")

In [4]:
# Calculate Net Machine Capacity (Available - WIP)
machine_net_cap = {}
machine_rates = {}
for _, row in df_machines.iterrows():
    m_id = row['machine_id']
    machine_net_cap[m_id] = max(0, row['available_weekly_hours'] - row['existing_wip_hours'])
    machine_rates[m_id] = row['operating_cost_per_hour']

In [5]:
# True Cost to Buy = Vendor Price + Incoming QA Cost
# (No LD penalty — vendors are assumed to deliver within the delivery period)
df['true_buy_cost'] = df['vendor_price_per_unit'] + df['incoming_qa_cost_per_unit']

In [6]:
# 3. Initialize the PuLP MILP Model
model = pulp.LpProblem("IOL_Make_vs_Buy_Optimization", pulp.LpMinimize)

In [7]:
# Decision Variables
# make_vars: how many units of each order to manufacture in-house (integer)
# buy_vars:  how many units of each order to buy externally (integer)
# is_make:   binary indicator — 1 if ANY units are made in-house for this order
#            (needed to model one-time setup hours/costs on machines)

make_vars = pulp.LpVariable.dicts("Make", df['order_id'], lowBound=0, cat='Integer')
buy_vars  = pulp.LpVariable.dicts("Buy",  df['order_id'], lowBound=0, cat='Integer')
is_make   = pulp.LpVariable.dicts("IsMake", df['order_id'], cat='Binary')

In [8]:
# Link binary indicator to make quantity:
#   make_vars[order] <= order_quantity * is_make[order]
#   make_vars[order] >= 1 * is_make[order]  (if is_make=1, must make at least 1)
# This ensures is_make=1 exactly when make_vars > 0.

for _, row in df.iterrows():
    order = row['order_id']
    qty = row['order_quantity']
    model += make_vars[order] <= qty * is_make[order], f"Link_Upper_{order}"
    model += make_vars[order] >= is_make[order],       f"Link_Lower_{order}"

In [9]:
# 4. Build Objective Function
objective_terms = []

for _, row in df.iterrows():
    order = row['order_id']
    
    # --- MAKE cost (per unit): Raw Material × (1 + Scrap%) ---
    rm_cost = row['raw_material_cost_per_unit'] * (1 + row['expected_scrap_percentage'])
    objective_terms.append(make_vars[order] * rm_cost)
    
    # --- BUY cost (per unit): Vendor Price + QA ---
    objective_terms.append(buy_vars[order] * row['true_buy_cost'])

In [10]:
# 5. Machine Capacity Constraints + Machine Operating Costs
#
# For each machine:
#   Total hours used = SUM over orders [
#       is_make[order] * setup_hours          (one-time setup, triggered by binary)
#     + make_vars[order] * run_hours * scrap   (per-unit run time)
#   ] <= Net Capacity
#
# Machine operating cost added to objective for both setup and run components.

for m_id in machine_net_cap.keys():
    machine_usage = []
    
    for _, row in df.iterrows():
        order = row['order_id']
        item = row['item_id']
        scrap_factor = 1 + row['expected_scrap_percentage']

        # Find routing step for this item on this machine
        routing_step = df_routings[
            (df_routings['item_id'] == item) & (df_routings['machine_id'] == m_id)
        ]
        
        if not routing_step.empty:
            run_hrs   = routing_step['run_hours_per_unit'].values[0]
            setup_hrs = routing_step['setup_hours'].values[0]
            rate      = machine_rates[m_id]

            # --- Capacity usage ---
            # One-time setup (only if we decide to make)
            machine_usage.append(is_make[order] * setup_hrs)
            # Per-unit run time (adjusted for scrap)
            machine_usage.append(make_vars[order] * run_hrs * scrap_factor)

            # --- Operating cost in objective ---
            # One-time setup cost
            objective_terms.append(is_make[order] * setup_hrs * rate)
            # Per-unit run cost
            objective_terms.append(make_vars[order] * run_hrs * scrap_factor * rate)

    if machine_usage:
        model += pulp.lpSum(machine_usage) <= machine_net_cap[m_id], f"CapLimit_{m_id}"

In [11]:
# Set the complete objective function
model += pulp.lpSum(objective_terms)

In [12]:
# 6. Demand & Strategic Constraints
for _, row in df.iterrows():
    order = row['order_id']

    # Constraint: Fulfill total demand for the order
    model += make_vars[order] + buy_vars[order] == row['order_quantity'], f"Demand_{order}"

    # Constraint: Critical items MUST be made in-house
    if row['is_critical_component']:
        model += buy_vars[order] == 0, f"Strategic_Make_{order}"

In [13]:
# 7. Solve the Model
model.solve()

1

In [14]:
# 8. Extract Results
results = []
for _, row in df.iterrows():
    order = row['order_id']
    make_qty = make_vars[order].varValue
    buy_qty  = buy_vars[order].varValue
    
    results.append({
        'Order_ID': order,
        'Item_ID': row['item_id'],
        'Demand': row['order_quantity'],
        'Make_In_House_Qty': make_qty,
        'Buy_External_Qty': buy_qty,
        'Is_Critical': row['is_critical_component'],
        'Decision': 'MAKE' if buy_qty == 0 else ('BUY' if make_qty == 0 else 'SPLIT')
    })

In [15]:
df_results = pd.DataFrame(results)
df_results.to_csv("Final_Make_Buy_Decisions.csv", index=False)

In [16]:
# 9. Summary
print("Optimization Complete. Status:", pulp.LpStatus[model.status])
print(f"Optimal Total Cost: \u20b9{pulp.value(model.objective):,.2f}")
print()
print("Decision Summary:")
print(df_results['Decision'].value_counts().to_string())
print(f"\nTotal MAKE qty: {df_results['Make_In_House_Qty'].sum():.0f}")
print(f"Total BUY  qty: {df_results['Buy_External_Qty'].sum():.0f}")
print("\nResults saved to Final_Make_Buy_Decisions.csv")

Optimization Complete. Status: Optimal
Optimal Total Cost: ₹135,974,996.12

Decision Summary:
Decision
BUY     70
MAKE    30

Total MAKE qty: 3803
Total BUY  qty: 6695

Results saved to Final_Make_Buy_Decisions.csv
